In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return 100 * np.mean(
        2 * np.abs(y_pred - y_true) /
        (np.abs(y_true) + np.abs(y_pred))
    )

def mase(y_true, y_pred, y_train):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_train = np.array(y_train)

    naive_error = np.mean(np.abs(np.diff(y_train)))
    return np.mean(np.abs(y_true - y_pred)) / (naive_error)

def theil_u(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    denominator = np.sqrt(np.mean(y_true ** 2) + np.mean(y_pred ** 2))
    u = rmse / denominator
    return u

# Loading the data
df = pd.read_excel('output_week.xlsx')

# Count values and sort by week
data = df['WEEK'].value_counts().reset_index()
data.columns = ['WEEK', 'DATA']

# Convert week to a date representing the start of the week
data['WEEK_period'] = pd.to_datetime(data['WEEK'] + '-1', format='%G-%V-%u', errors='coerce')
data = data.sort_values('WEEK_period').reset_index(drop=True)

# ------------------------------
# Smoothing (moving average)
# ------------------------------
data['DATA_smooth'] = data['DATA'].rolling(window=5, center=True).mean()
data['DATA_smooth'].fillna(method='bfill', inplace=True)
data['DATA_smooth'].fillna(method='ffill', inplace=True)

# Log transformation
data['y'] = np.log1p(data[['DATA_smooth']])

/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1961/3642630466.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['DATA_smooth'].fillna(method='bfill', inplace=True)
/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1961/3642630466.py:45: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['DATA_smooth'].fillna(method='bfill', inplace=True)
/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1961/364263046

In [3]:
# ------------------------------
# Time Series Cross Validation
# ------------------------------

tscv = TimeSeriesSplit(n_splits=10)

rmse_sarima = []
mae_sarima = []
mase_sarima = []
smape_sarima = []


for fold, (train_idx, test_idx) in enumerate(tscv.split(data["y"]), 1):

    print(f"\nFold {fold}")


    # ------------------------------
    # Split temporal
    # ------------------------------

    y_train = data["y"].iloc[train_idx]
    y_test = data["y"].iloc[test_idx]


    # ------------------------------
    # SARIMA
    # ------------------------------

    model_sarima = SARIMAX(
        y_train,
        order=(5, 1, 4),
        seasonal_order=(0, 1, 1, 53),
        enforce_stationarity=False,
        enforce_invertibility=False
    )


    model_fit_sarima = model_sarima.fit(
        disp=False
    )


    # ------------------------------
    # Forecast
    # ------------------------------

    y_pred = model_fit_sarima.forecast(
        steps=len(y_test)
    )


    # ------------------------------
    # Retornar escala original
    # ------------------------------

    y_train_real = np.expm1(
        y_train.to_numpy()
    )

    y_test_real = np.expm1(
        y_test.to_numpy()
    )

    y_pred_real = np.expm1(
        np.asarray(y_pred)
    )


    # ------------------------------
    # Métricas
    # ------------------------------

    rmse_value = np.sqrt(
        mean_squared_error(
            y_test_real,
            y_pred_real
        )
    )


    mae_value = mean_absolute_error(
        y_test_real,
        y_pred_real
    )


    mase_value = mase(
        y_test_real,
        y_pred_real,
        y_train_real
    )


    smape_value = smape(
        y_test_real,
        y_pred_real
    )


    # ------------------------------
    # Guardar resultados
    # ------------------------------

    rmse_sarima.append(rmse_value)
    mae_sarima.append(mae_value)
    mase_sarima.append(mase_value)
    smape_sarima.append(smape_value)


    print(
        f"RMSE={rmse_value:.4f} | "
        f"MAE={mae_value:.4f}"
    )


# ------------------------------
# Resultados finais
# ------------------------------

results_sarima = pd.DataFrame({

    "Fold": range(1, len(rmse_sarima)+1),
    "RMSE": rmse_sarima,
    "MAE": mae_sarima,
    "MASE": mase_sarima,
    "SMAPE": smape_sarima

})


results_sarima.to_csv(
    "SARIMA_results.csv",
    index=False
)


Fold 1


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'


RMSE=143.9386 | MAE=113.7054

Fold 2


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'


RMSE=308.2550 | MAE=93.8678

Fold 3


...

RMSE=5.0041 | MAE=4.5877

Fold 9


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


RMSE=4.1337 | MAE=3.6790

Fold 10


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


RMSE=0.9484 | MAE=0.7247
